# Can we predict a warranty claim within a year?
Target: `claim_within_year`. Prediction time: when the warranty is sold.

Design decisions (from checklist.md and columns.csv):
- `device_serial` is an identifier: dropped. It is unique per row (checked below), so no entity grouping is needed.
- There is no date column, so no time ordering is available; the test set is a random stratified hold-out (20%), set aside before any modelling. If row order were chronological this would be optimistic; we cannot verify that.
- All remaining columns (channel, device age, price, daily hours at registration) are recorded at sale/registration, so they are available at prediction time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
df = pd.read_csv('data.csv')
print(df.shape, 'unique serials:', df.device_serial.nunique(), 'missing:', int(df.isna().sum().sum()))
print(df.claim_within_year.mean())

## 1. Set aside the test set (before any modelling)

In [ ]:
TARGET='claim_within_year'
X = df.drop(columns=[TARGET,'device_serial']); y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
print(len(X_train), len(X_test), y_train.mean(), y_test.mean())

## 2. Candidate models (named before scoring)
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg`: logistic regression (scaled numerics, one-hot channel), C in {0.1, 1, 10}.
3. `hgb`: histogram gradient boosting, small grid over learning rate / depth.
4. `rf`: random forest, small grid over min_samples_leaf.

Selection metric: cross-validated ROC AUC (PR AUC reported too, since the positive rate is ~13%). Decision threshold: maximise F1 on out-of-fold predictions of the chosen model. Preprocessing sits inside the pipelines so it is fit per fold. 5-fold stratified CV on training rows only (rows are unique entities, so whole-entity folds hold automatically).

In [ ]:
num=['device_age_years','price_eur','daily_hours']; cat=['channel']
def prep(scale):
    return ColumnTransformer([('n', StandardScaler() if scale else 'passthrough', num),
                              ('c', OneHotEncoder(handle_unknown='ignore'), cat)])
def mk(name, **p):
    if name=='baseline': return Pipeline([('m',DummyClassifier(strategy='prior'))])
    if name=='logreg': return Pipeline([('p',prep(True)),('m',LogisticRegression(C=p['C'],max_iter=1000))])
    if name=='hgb': return Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(learning_rate=p['lr'],max_depth=p['d'],max_iter=150,random_state=0))])
    if name=='rf': return Pipeline([('p',prep(False)),('m',RandomForestClassifier(n_estimators=300,min_samples_leaf=p['leaf'],random_state=0,n_jobs=-1))])
cands = [('baseline',{})]+[('logreg',{'C':C}) for C in (0.1,1,10)] \
      +[('hgb',{'lr':lr,'d':d}) for lr in (0.03,0.1) for d in (2,3)] \
      +[('rf',{'leaf':l}) for l in (5,20,50)]
cv = StratifiedKFold(5, shuffle=True, random_state=1)
rows=[]; oof={}
for name,p in cands:
    pr = cross_val_predict(mk(name,**p), X_train, y_train, cv=cv, method='predict_proba')[:,1]
    key=f'{name} {p}'; oof[key]=pr
    rows.append((key, roc_auc_score(y_train,pr), average_precision_score(y_train,pr)))
cvres = pd.DataFrame(rows, columns=['model','cv_auc','cv_pr_auc']).sort_values('cv_auc',ascending=False)
cvres

## 3. Choose model and threshold on CV only

In [ ]:
best_key = cvres.iloc[0].model
best_name, best_p = next((n,p) for n,p in cands if f'{n} {p}'==best_key)
pr = oof[best_key]
ths = np.linspace(0.02,0.8,157)
f1s = [f1_score(y_train, pr>=t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('chosen:', best_key, '| CV AUC %.3f'%cvres.iloc[0].cv_auc, '| threshold %.3f'%thr, '| CV F1 %.3f'%max(f1s))
print('baseline CV AUC:', cvres[cvres.model.str.startswith('baseline')].cv_auc.iloc[0])

**Choice recorded before test scoring:** the model above (top CV ROC AUC), with the F1-maximising out-of-fold threshold printed above. The test set has not been touched so far.

## 4. Refit on all training rows; score the test set once

In [ ]:
final = mk(best_name, **best_p).fit(X_train, y_train)
pt = final.predict_proba(X_test)[:,1]
pred = pt>=thr
print('TEST ROC AUC   : %.3f'%roc_auc_score(y_test,pt))
print('TEST PR AUC    : %.3f (prevalence %.3f)'%(average_precision_score(y_test,pt), y_test.mean()))
print('TEST precision : %.3f  recall: %.3f  F1: %.3f at threshold %.3f'%(precision_score(y_test,pred),recall_score(y_test,pred),f1_score(y_test,pred),thr))
print('Trivial baseline AUC = 0.5; always-positive F1 = %.3f'%f1_score(y_test,np.ones(len(y_test))))

## Conclusion
The printed test metrics are the headline (single scoring of the held-out set). Compare the test ROC AUC with 0.5 (baseline) and the PR AUC with the prevalence. Caveats: no date column, so the split is random rather than temporal; one test set of ~1,400 rows (~176 positives) gives noisy estimates.